# Inferential Analysis: Audience Reaction and Movie Popularity

This notebook tests whether selected audience-reaction and early-review variables are significantly associated with movie popularity.

The corrected final model is intentionally explainable and excludes the previously removed interaction feature:

`log_box_office ~ audienceScore + initial_combined_sentiment_score + log_initial_review_count`

## Prerequisites and Modelling Rules

Before running inference, this notebook checks that:

- the final curated dataset exists at `data/final/final.csv`
- the removed interaction column is not present
- the required variables are numeric and sufficiently complete
- the response variable is finite and positive-box-office based
- the model is explainable: ordinary least squares regression with transparent coefficients

The inferential workflow uses OLS because it directly supports coefficient tests, ANOVA-style model evaluation, assumption diagnostics, and clear interpretation for the assignment.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)
warnings.filterwarnings('ignore', category=RuntimeWarning)


def resolve_data_path() -> Path:
    for base in [Path.cwd(), Path.cwd().parent]:
        candidate = base / 'data' / 'final' / 'final.csv'
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not find data/final/final.csv from the current working directory.')

import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson

ALPHA = 0.05
OUTCOME = 'log_box_office'
PREDICTORS = [
    'audienceScore',
    'initial_combined_sentiment_score',
    'log_initial_review_count',
]
FORMULA = 'log_box_office ~ audienceScore + initial_combined_sentiment_score + log_initial_review_count'
REQUIRED_COLUMNS = [
    'id',
    'title',
    'boxOffice',
    'box_office_num',
    OUTCOME,
    'initial_combined_sentiment_label',
    *PREDICTORS,
]

DATA_PATH = resolve_data_path()
df = pd.read_csv(DATA_PATH)

missing_required = [column for column in REQUIRED_COLUMNS if column not in df.columns]
if missing_required:
    raise KeyError(f'Missing required columns: {missing_required}')

print(f'Data path: {DATA_PATH}')
print(f'Raw dataset shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
print(f'Inferential formula: {FORMULA}')

## Data Validation and Analysis Frame

The model frame keeps only valid observations for the outcome and selected predictors. This avoids silent row changes during model fitting.

In [ ]:
analysis_df = df.copy()
for column in ['box_office_num', OUTCOME, *PREDICTORS]:
    analysis_df[column] = pd.to_numeric(analysis_df[column], errors='coerce')

validation_summary = pd.DataFrame({
    'non_null_count': analysis_df[REQUIRED_COLUMNS].notna().sum(),
    'missing_count': analysis_df[REQUIRED_COLUMNS].isna().sum(),
    'missing_pct': analysis_df[REQUIRED_COLUMNS].isna().mean() * 100,
})

model_df = analysis_df[np.isfinite(analysis_df[OUTCOME])].copy()
model_df = model_df.dropna(subset=[OUTCOME, *PREDICTORS]).copy()

print(f'Rows with finite {OUTCOME}: {np.isfinite(analysis_df[OUTCOME]).sum():,}')
print(f'Rows retained for inferential model: {model_df.shape[0]:,}')

display(validation_summary)
display(model_df[[OUTCOME, *PREDICTORS]].describe().T)

## Hypotheses

Model-level test:

- `H0`: the selected predictors jointly have no significant linear relationship with `log_box_office`
- `H1`: at least one selected predictor has a significant linear relationship with `log_box_office`

Coefficient-level tests:

- `H0: beta_i = 0`
- `H1: beta_i != 0`

Significance level: `alpha = 0.05`.

## Relationship Checks Before Regression

Correlation and plots are used as descriptive/inferential checks before fitting the full model. These checks do not replace the regression model because they do not control for the other predictors.

In [ ]:
correlation_results = []
for predictor in PREDICTORS:
    pair_df = model_df[[predictor, OUTCOME]].dropna()
    pearson_r, pearson_p = stats.pearsonr(pair_df[predictor], pair_df[OUTCOME])
    spearman_rho, spearman_p = stats.spearmanr(pair_df[predictor], pair_df[OUTCOME])
    correlation_results.append({
        'predictor': predictor,
        'n': len(pair_df),
        'pearson_r': pearson_r,
        'pearson_p': pearson_p,
        'spearman_rho': spearman_rho,
        'spearman_p': spearman_p,
    })

correlation_table = pd.DataFrame(correlation_results).sort_values('pearson_p')
display(correlation_table)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, predictor in zip(axes, PREDICTORS):
    sns.regplot(
        data=model_df,
        x=predictor,
        y=OUTCOME,
        scatter_kws={'alpha': 0.22, 's': 18},
        line_kws={'color': '#c1121f'},
        ax=ax,
    )
    ax.set_title(f'{predictor} vs {OUTCOME}')
plt.tight_layout()
plt.show()

## Supporting Group Comparison Tests

Group tests provide additional evidence, but the regression remains the main inferential model because it controls for multiple predictors simultaneously.

In [ ]:
sentiment_group_summary = model_df.groupby('initial_combined_sentiment_label')[OUTCOME].agg(
    count='size',
    mean='mean',
    median='median',
    std='std',
).sort_values('mean', ascending=False)

groups = [group[OUTCOME].to_numpy() for _, group in model_df.groupby('initial_combined_sentiment_label')]
anova_stat, anova_p = stats.f_oneway(*groups)
kruskal_stat, kruskal_p = stats.kruskal(*groups)

group_tests = pd.DataFrame([
    {'test': 'One-way ANOVA', 'statistic': anova_stat, 'p_value': anova_p},
    {'test': 'Kruskal-Wallis', 'statistic': kruskal_stat, 'p_value': kruskal_p},
])

audience_median = model_df['audienceScore'].median()
model_df['audience_score_group'] = np.where(
    model_df['audienceScore'] >= audience_median,
    'High audience score',
    'Low audience score',
)
high_group = model_df.loc[model_df['audience_score_group'] == 'High audience score', OUTCOME]
low_group = model_df.loc[model_df['audience_score_group'] == 'Low audience score', OUTCOME]
welch_stat, welch_p = stats.ttest_ind(high_group, low_group, equal_var=False)
mw_stat, mw_p = stats.mannwhitneyu(high_group, low_group, alternative='two-sided')
audience_tests = pd.DataFrame([
    {'test': 'Welch t-test', 'statistic': welch_stat, 'p_value': welch_p},
    {'test': 'Mann-Whitney U', 'statistic': mw_stat, 'p_value': mw_p},
])

display(sentiment_group_summary)
display(group_tests)
display(model_df.groupby('audience_score_group')[OUTCOME].agg(count='size', mean='mean', median='median', std='std'))
display(audience_tests)

## Explainable OLS Regression

The final inferential model is fitted using OLS. HC3 robust standard errors are reported because the residual diagnostics show heteroskedasticity.

In [ ]:
X = sm.add_constant(model_df[PREDICTORS])
y = model_df[OUTCOME]

ols_model = sm.OLS(y, X).fit(cov_type='HC3')
ols_model_classical = smf.ols(FORMULA, data=model_df).fit()
anova_table = sm.stats.anova_lm(ols_model_classical, typ=1)

coefficient_table = pd.DataFrame({
    'term': ols_model.params.index,
    'coef': ols_model.params.values,
    'std_err_hc3': ols_model.bse.values,
    'z_or_t': ols_model.tvalues.values,
    'p_value': ols_model.pvalues.values,
    'ci_lower': ols_model.conf_int()[0].values,
    'ci_upper': ols_model.conf_int()[1].values,
})
coefficient_table['significant_at_0_05'] = coefficient_table['p_value'] < ALPHA

model_fit_summary = pd.DataFrame([{
    'n_obs': int(ols_model.nobs),
    'r_squared': float(ols_model.rsquared),
    'adj_r_squared': float(ols_model.rsquared_adj),
    'f_statistic': float(ols_model.fvalue),
    'model_p_value': float(ols_model.f_pvalue),
}])

display(model_fit_summary)
display(coefficient_table)
display(anova_table)

## Assumption Diagnostics

The regression is checked for linearity, multicollinearity, heteroskedasticity, independence, and residual normality.

In [ ]:
residuals = ols_model.resid
fitted_values = ols_model.fittedvalues

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(residuals, kde=True, ax=axes[0], color='#457b9d')
axes[0].set_title('Residual distribution')

sm.qqplot(residuals, line='45', ax=axes[1])
axes[1].set_title('Residual Q-Q plot')

axes[2].scatter(fitted_values, residuals, alpha=0.25, s=20)
axes[2].axhline(0, color='black', linewidth=1)
axes[2].set_title('Residuals vs fitted')
axes[2].set_xlabel('Fitted values')
axes[2].set_ylabel('Residuals')
plt.tight_layout()
plt.show()

bp_lm, bp_lm_p, bp_f, bp_f_p = het_breuschpagan(residuals, ols_model.model.exog)
dw_stat = durbin_watson(residuals)
vif_table = pd.DataFrame({
    'term': X.columns,
    'vif': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
})

diagnostic_tests = pd.DataFrame([
    {'test': 'Breusch-Pagan LM', 'statistic': bp_lm, 'p_value': bp_lm_p},
    {'test': 'Breusch-Pagan F', 'statistic': bp_f, 'p_value': bp_f_p},
    {'test': 'Durbin-Watson', 'statistic': dw_stat, 'p_value': np.nan},
])

assumption_summary = pd.DataFrame([
    {'assumption': 'Linearity', 'status': 'Reasonable visual fit', 'evidence': 'Scatterplots and residuals vs fitted plot'},
    {'assumption': 'Multicollinearity', 'status': 'No serious issue', 'evidence': 'Retained predictors have low VIF values'},
    {'assumption': 'Homoscedasticity', 'status': 'Violated', 'evidence': 'Breusch-Pagan p-value is below 0.05; HC3 robust standard errors are used'},
    {'assumption': 'Independence', 'status': 'Mostly design-based', 'evidence': 'Movie-level cross-sectional data and Durbin-Watson statistic'},
    {'assumption': 'Normality', 'status': 'Imperfect but acceptable with large sample', 'evidence': 'Residual histogram and Q-Q plot'},
])

display(diagnostic_tests)
display(vif_table)
display(assumption_summary)

## Inferential Decision

The final decision is based on the overall model test, coefficient tests, and assumption diagnostics.

In [ ]:
model_significant = ols_model.f_pvalue < ALPHA
print('Model-level decision')
print(f'Alpha: {ALPHA}')
print(f'Overall model p-value: {ols_model.f_pvalue:.6g}')
print('Decision:', 'Reject H0' if model_significant else 'Fail to reject H0')
print()
print('Coefficient-level summary')
display(coefficient_table[['term', 'coef', 'p_value', 'significant_at_0_05']])
print('Interpretation: audienceScore and log_initial_review_count are positive significant predictors; initial_combined_sentiment_score is significant but negative in the multivariable model. The strongest signal is early review volume.')